# Rolling track synthesis

Parents: [25-research-rolling-metrics-controls.ipynb](25-research-rolling-metrics-controls.ipynb), [26-research-rolling-ranking-access.ipynb](26-research-rolling-ranking-access.ipynb), [27-research-rolling-sizing.ipynb](27-research-rolling-sizing.ipynb), [28-research-rolling-exposure.ipynb](28-research-rolling-exposure.ipynb).

## Results

This artefact-only notebook executed successfully after NB27/NB28 ledgers were populated. It writes lineage, all metrics and equity curves, exposure/cash, contributor and selection-coverage reports under `_artifacts-rolling-profit-risk/nb29`. It adds no search or simulation. Optional 14/60-day paired lookback and largest-contributor removal sensitivities are explicitly labelled unrun.

## Interpretation limit

The three windows overlap and reuse history, so they are not independent holdouts. The synthesis separates ranking, sizing and exposure effects rather than selecting a winning formula.


In [1]:

from pathlib import Path
import json
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

PROJECT = Path.cwd()
while not (PROJECT / 'pyproject.toml').exists():
    if PROJECT == PROJECT.parent:
        raise RuntimeError('Repository root not found')
    PROJECT = PROJECT.parent
PROJECT = PROJECT / 'scratchpad/hyperliquid-ic'
OUT = PROJECT / '_artifacts-rolling-profit-risk' / 'nb29'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT))
from rolling_track_reporting import read_optional_csv, write_pending_manifest

ARTIFACT_ROOT = PROJECT / '_artifacts-rolling-profit-risk'


In [2]:

metrics_frames = []
curve_frames = []
position_frames = []
allocation_frames = []
lineage_rows = []
for number in (25, 26, 27, 28):
    directory = ARTIFACT_ROOT / f'nb{number}'
    manifest_path = directory / 'manifest.json'
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    status = manifest.get('status', 'executed' if (directory / 'portfolio-metrics.csv').exists() else 'missing')
    lineage_rows.append({
        'notebook': number,
        'status': status,
        'parent': manifest.get('parent_experiment'),
        'mechanism_delta': manifest.get('mechanism_delta'),
        'metrics_rows': len(read_optional_csv(directory / 'portfolio-metrics.csv')),
    })
    metrics = read_optional_csv(directory / 'portfolio-metrics.csv')
    if not metrics.empty:
        metrics.insert(0, 'notebook', number)
        metrics_frames.append(metrics)
    positions = read_optional_csv(directory / 'positions.csv')
    if not positions.empty:
        positions.insert(0, 'notebook', number)
        position_frames.append(positions)
    allocations = read_optional_csv(directory / 'allocations.csv')
    if not allocations.empty:
        allocations.insert(0, 'notebook', number)
        allocation_frames.append(allocations)
    for curve_path in sorted(directory.glob('curve-*.parquet')):
        curve = pd.read_parquet(curve_path).reset_index()
        curve.columns = ['date', 'equity']
        stem = curve_path.stem.split('-', 2)
        curve['notebook'] = number
        curve['period'] = stem[1] if len(stem) > 1 else None
        curve['arm'] = stem[2] if len(stem) > 2 else None
        curve_frames.append(curve)

lineage = pd.DataFrame(lineage_rows)
metrics = pd.concat(metrics_frames, ignore_index=True) if metrics_frames else pd.DataFrame()
curves = pd.concat(curve_frames, ignore_index=True) if curve_frames else pd.DataFrame()
positions = pd.concat(position_frames, ignore_index=True) if position_frames else pd.DataFrame()
allocations = pd.concat(allocation_frames, ignore_index=True) if allocation_frames else pd.DataFrame()
lineage.to_csv(OUT / 'lineage-results.csv', index=False)
metrics.to_csv(OUT / 'metrics-all.csv', index=False)
curves.to_csv(OUT / 'equity-curves-long.csv', index=False)

if not curves.empty:
    curves['series'] = curves['notebook'].astype(str) + ':' + curves['period'].astype(str) + ':' + curves['arm'].astype(str)
    fig = px.line(curves, x='date', y='equity', color='series', title='Rolling track equity curves')
    fig.write_html(OUT / 'equity-curves.html', include_plotlyjs='cdn')

if not allocations.empty:
    exposure = allocations.groupby(['notebook', 'period', 'arm']).agg(
        cycles=('date', 'size'),
        mean_invested=('invested', 'mean'),
        min_invested=('invested', 'min'),
        mean_positions=('positions', 'mean'),
        mean_max_weight=('max_weight', 'mean'),
    ).reset_index()
else:
    exposure = pd.DataFrame()
exposure.to_csv(OUT / 'exposure-report.csv', index=False)

if not positions.empty:
    contributor = positions.groupby(['notebook', 'period', 'arm', 'address', 'name'], dropna=False).agg(
        position_count=('position_id', 'nunique'), pnl=('pnl', 'sum')
    ).reset_index()
    contributor['abs_pnl_share'] = contributor['pnl'].abs() / contributor.groupby(['notebook', 'period', 'arm'])['pnl'].transform(lambda x: x.abs().sum()).replace(0, float('nan'))
else:
    contributor = pd.DataFrame()
contributor.to_csv(OUT / 'contributor-report.csv', index=False)

replay = read_optional_csv(ARTIFACT_ROOT / 'nb27' / 'schedule-replay-log.csv')
if not replay.empty:
    replay_summary = replay.groupby('arm').agg(cycles=('date', 'size'), mean_jaccard=('jaccard', 'mean'), missing=('missing_count', 'sum'), unexpected=('unexpected_count', 'sum')).reset_index()
else:
    replay_summary = pd.DataFrame()
replay_summary.to_csv(OUT / 'selection-coverage-report.csv', index=False)

ranking = read_optional_csv(ARTIFACT_ROOT / 'nb26' / 'rolling-ranking-summary.csv')
ranking.to_csv(OUT / 'selection-before-label-coverage.csv', index=False)

deferred = pd.DataFrame([
    {'sensitivity': 'B01/B11 interval lookback 14 and 60 days', 'status': 'declared_not_run', 'reason': 'bounded robustness follow-up after primary results'},
    {'sensitivity': 'largest positive B11 contributor removal', 'status': 'declared_not_run', 'reason': 'retrospective sensitivity; identity fixed from primary result'},
])
deferred.to_csv(OUT / 'deferred-robustness.csv', index=False)
display(lineage)
display(metrics)
display(exposure)
display(replay_summary)
write_pending_manifest(OUT, notebook='29-research-rolling-track-synthesis.ipynb', parent='27-research-rolling-sizing.ipynb and 28-research-rolling-exposure.ipynb', mechanism_delta='Artefact-only synthesis of validated rolling-track experiments.', inputs=['nb25', 'nb26', 'nb27', 'nb28'], status='executed', why_not_duplicate='No new simulation, optimiser or result-dependent selection; all tables and curves are derived from saved experiment artefacts.')


,notebook,status,parent,mechanism_delta,metrics_rows
0,25,executed,24-research-monthly-calibration.ipynb,Direct rolling observed-price features plus le...,6
1,26,executed,25-research-rolling-metrics-controls.ipynb,Four-arm ranking/access factorial: original co...,12
2,27,executed,26-research-rolling-ranking-access.ipynb,Fixed causal B11 requested-membership replay w...,9
3,28,executed,25-research-rolling-metrics-controls.ipynb,B00 parent with causal no-refill and drawdown-...,12


,notebook,period,arm,cagr,sharpe,weekly_sharpe,volatility,max_drawdown,best_cycle,kurtosis,start,end
0,25,hyper_ai,LEGACY_hyper_ai,0.412339,2.005386,2.131391,0.180191,-0.101862,0.075309,9.924390,2026-01-01,2026-07-08
1,25,full,LEGACY_full,0.047707,0.343223,0.338858,0.186153,-0.138054,0.073447,11.569952,2025-09-13,2026-09-08
2,25,later,LEGACY_later,0.189981,1.005850,0.984320,0.190564,-0.086945,0.070357,6.992972,2026-04-01,2026-09-08
3,25,hyper_ai,B00,0.515094,2.382880,2.548373,0.181250,-0.082629,0.077237,10.578121,2026-01-01,2026-07-08
4,25,full,B00,0.097268,0.584833,0.571515,0.189284,-0.138234,0.075257,11.292243,2025-09-13,2026-09-08
5,25,later,B00,0.321314,1.518765,1.453242,0.195870,-0.069413,0.072118,6.712677,2026-04-01,2026-09-08
6,26,hyper_ai,B00,0.515094,2.382880,2.548373,0.181250,-0.082629,0.077237,10.578121,2026-01-01,2026-07-08
7,26,hyper_ai,B10,-0.723703,-1.911989,-1.890141,0.581223,-0.570605,0.196793,8.940920,2026-01-01,2026-07-08
8,26,hyper_ai,B01,0.505343,2.392338,2.653681,0.177566,-0.083998,0.077291,11.068523,2026-01-01,2026-07-08
9,26,hyper_ai,B11,-0.872856,-2.663056,-2.462862,0.682098,-0.717460,0.255660,12.522059,2026-01-01,2026-07-08


,notebook,period,arm,cycles,mean_invested,min_invested,mean_positions,mean_max_weight
0,25,full,B00,203,0.957803,0.0,5.911330,0.315654
1,25,full,LEGACY_full,203,0.959811,0.0,5.916256,0.315387
2,25,hyper_ai,B00,95,0.964808,0.0,5.842105,0.314993
3,25,hyper_ai,LEGACY_hyper_ai,95,0.966438,0.0,5.852632,0.314675
4,25,later,B00,81,0.964634,0.0,5.814815,0.313304
5,25,later,LEGACY_later,81,0.967025,0.0,5.827160,0.314071
6,26,full,B00,203,0.957803,0.0,5.911330,0.315654
7,26,full,B01,203,0.953632,0.0,5.832512,0.316170
8,26,full,B10,203,0.588346,0.0,4.275862,0.255752
9,26,full,B11,203,0.898646,0.0,5.970443,0.301752


,arm,cycles,mean_jaccard,missing,unexpected
0,S_EQUAL,379,1.0,0,0
1,S_INV_VAR,379,1.0,0,0
2,S_INV_VOL,379,1.0,0,0
